In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import Ridge
import warnings
import pickle
import os

warnings.filterwarnings('ignore')

print("===  PIPELINE  ===")


train_path = 'data/train.csv' 
test_path = 'data/test.csv'

train = pd.read_csv(train_path, sep=';', decimal=',', low_memory=False)
test = pd.read_csv(test_path, sep=';', decimal=',', low_memory=False)

print(f"Train: {train.shape}, Test: {test.shape}")

===  PIPELINE  ===
Train: (76786, 224), Test: (73214, 222)


In [2]:
# временные признаки
def make_time_features(df):
    df = df.copy()
    dt = pd.to_datetime(df['dt'])
    start_date = dt.min()
    time_df = pd.DataFrame({
        'days_from_start': (dt - start_date).dt.days,
        'month_sin': np.sin(2 * np.pi * dt.dt.month / 12),
        'month_cos': np.cos(2 * np.pi * dt.dt.month / 12),
        'dayofyear_sin': np.sin(2 * np.pi * dt.dt.dayofyear / 365.25),
        'dayofyear_cos': np.cos(2 * np.pi * dt.dt.dayofyear / 365.25),
        'quarter': dt.dt.quarter,
        'is_weekend': dt.dt.dayofweek.isin([5, 6]).astype(int),
    })
    return pd.concat([df.drop(columns=['dt']), time_df], axis=1)

train = make_time_features(train)
test = make_time_features(test)
test_id = test['id'].copy()


y = train['target'].values
w = train['w'].values
X = train.drop(['target', 'w', 'id'], axis=1)
X_test = test.drop(['id'], axis=1)

#FEATURE ENGINEERING
def advanced_feature_engineering(df):
    df = df.copy()
    
    # основные ratios
    if 'avg_credit_turn_rur' in df.columns and 'avg_debet_turn_rur' in df.columns:
        df['credit_debit_ratio'] = df['avg_credit_turn_rur'] / (df['avg_debet_turn_rur'] + 1e-8)
        df['log_credit_ratio'] = np.log1p(df['credit_debit_ratio'])
    
    #агрегаты по категориям трат
    cat_cols = [c for c in df.columns if c.startswith('avg_by_category__amount__sum__') or 
                c.startswith('by_category__amount__sum__')]
    if cat_cols:
        df['total_cat_spend_mean'] = df[cat_cols].mean(axis=1)
        df['total_cat_spend_sum'] = df[cat_cols].sum(axis=1)
        df['total_cat_spend_std'] = df[cat_cols].std(axis=1)
        df['total_cat_spend_max'] = df[cat_cols].max(axis=1)
    
    #  Обороты
    cr_cols = [c for c in df.columns if 'cr_' in c and 'turn' in c]
    db_cols = [c for c in df.columns if 'db_' in c and 'turn' in c]
    if cr_cols and db_cols:
        df['total_cr_turn'] = df[cr_cols].sum(axis=1, skipna=True)
        df['total_db_turn'] = df[db_cols].sum(axis=1, skipna=True)
        df['net_turnover'] = df['total_cr_turn'] - df['total_db_turn']
    
    # BKI aggregates
    limit_cols = [c for c in df.columns if 'max_limit' in c]
    if limit_cols:
        df['total_bki_limit'] = df[limit_cols].sum(axis=1, skipna=True)
    

    if 'days_from_start' in df.columns and 'age' in df.columns:
        df['age_time_interact'] = df['age'] * df['days_from_start']
        df['days_squared'] = df['days_from_start'] ** 2
    
    return df

X = advanced_feature_engineering(X)
X_test = advanced_feature_engineering(X_test)

print(f"После FE признаков: {X.shape[1]}")

После FE признаков: 239


In [3]:
#препроцессинг

cat_features = [col for col in X.columns if X[col].dtype == 'object' or 
                X[col].dtype.name == 'category' or 
                X[col].dtype == 'bool']

num_features = [col for col in X.columns if col not in cat_features]

def preprocess_data(df, cat_features, encoders=None):
    df = df.copy()
    
    # NaN indicators + fill для числовых
    for col in num_features:
        if col in df.columns:
            df[col + '_isna'] = df[col].isna().astype(int)
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-999)
    
    #категор.
    if encoders is None:
        encoders = {}
    for col in cat_features:
        if col in df.columns:
            df[col] = df[col].fillna('MISSING').astype(str)
            if col not in encoders:
                oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
                df[col] = oe.fit_transform(df[[col]]).ravel().astype(int)
                encoders[col] = oe
            else:
                df[col] = encoders[col].transform(df[[col]]).ravel().astype(int)
    
    return df, encoders

#обнова
X_proc, encoders = preprocess_data(X, cat_features)
X_test_proc, _ = preprocess_data(X_test, cat_features, encoders)

#что б потом не было проблем все в один тип длинный 
X_proc = X_proc.astype(np.float32)
X_test_proc = X_test_proc.astype(np.float32)

print(f"Предобработка завершена. Финальное кол-во признаков: {X_proc.shape[1]}")
print(f"Категориальных признаков было: {len(cat_features)}")

Предобработка завершена. Финальное кол-во признаков: 478
Категориальных признаков было: 0


In [ ]:
#  OPTUNA  подбор лучших параметров 
def objective(trial):
    params = {
        'objective': 'reg:absoluteerror',
        'tree_method': 'hist',
        'device': 'cpu',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 30),
        'subsample': trial.suggest_float('subsample', 0.7, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.95),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.5, 10.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 15.0),
        'random_state': 42,
        'verbosity': 0,
    }
    
    tscv = TimeSeriesSplit(n_splits=3)  
    scores = []
    
    for train_idx, val_idx in tscv.split(X_proc):
        X_tr = X_proc.iloc[train_idx]
        X_val = X_proc.iloc[val_idx]
        y_tr = y[train_idx]
        y_val = y[val_idx]
        w_tr = w[train_idx]
        w_val = w[val_idx]
        
        dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
        dval = xgb.DMatrix(X_val, label=y_val, weight=w_val)
        
        model = xgb.train(
            params, dtrain,
            num_boost_round=1200,           
            evals=[(dval, 'val')],
            early_stopping_rounds=80,       
            verbose_eval=False
        )
        
        pred_val = model.predict(dval)
        wmae = np.sum(w_val * np.abs(y_val - pred_val)) / np.sum(w_val)
        scores.append(wmae)
    
    return np.mean(scores)



print("Запуск Optuna ")
study = optuna.create_study(
    direction='minimize', 
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=15, show_progress_bar=True)   

print(f"\n BEST CV WMAE: {study.best_value:.4f}")
print("Лучшие параметры:", study.best_params)

In [ ]:
#Обучние финал
best_params = study.best_params
dtrain_full = xgb.DMatrix(X_proc, label=y, weight=w)
dtest = xgb.DMatrix(X_test_proc)

final_model = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=3500,
    verbose_eval=200
)


pred_final = final_model.predict(dtest)


os.makedirs('v', exist_ok=True)

submission = pd.DataFrame({'id': test_id, 'predict': pred_final})
submission.to_csv('v/submission_v_xgb_best.csv', sep=';', decimal='.', index=False)

with open('v/model_v5_xgb.pkl', 'wb') as f:
    pickle.dump(final_model, f)

print("all good )")

In [ ]:


best_params = {
    'objective': 'reg:absoluteerror',
    'tree_method': 'hist',
    'device': 'cpu',
    'learning_rate': 0.01695,
    'max_depth': 10,
    'min_child_weight': 27,
    'subsample': 0.9058,
    'colsample_bytree': 0.7562,
    'reg_alpha': 4.154,
    'reg_lambda': 2.628,
    'random_state': 42,
    'verbosity': 0,
}

print("Обучаем финальную модель (пытаемся )...")

dtrain_full = xgb.DMatrix(X_proc, label=y, weight=w)
dtest = xgb.DMatrix(X_test_proc)

final_model = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=5000,           
    verbose_eval=200
)


pred_final = final_model.predict(dtest)


import os
os.makedirs('v', exist_ok=True)

pd.DataFrame({'id': test_id, 'predict': pred_final}).to_csv(
    'v/submission_v_final.csv', sep=';', decimal='.', index=False
)

with open('v/model_v_final.pkl', 'wb') as f:
    pickle.dump(final_model, f)


print(f"Лучший CV score из Optuna: {study.best_value:.2f}")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
import pickle
import os
import warnings
warnings.filterwarnings('ignore')


In [ ]:
#OOF STACKING 
from sklearn.model_selection import TimeSeriesSplit

print("Обучаем OOF Stacking...")

tscv = TimeSeriesSplit(n_splits=5)
oof_xgb = np.zeros(len(X_proc))
oof_lgb = np.zeros(len(X_proc))
oof_hist = np.zeros(len(X_proc))

test_xgb = np.zeros((len(X_test_proc), 5))
test_lgb = np.zeros((len(X_test_proc), 5))
test_hist = np.zeros((len(X_test_proc), 5))

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_proc)):
    print(f"Fold {fold+1}/5")
    
    X_tr, X_val = X_proc.iloc[train_idx], X_proc.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    w_tr, w_val = w[train_idx], w[val_idx]
    
    # XGBoost
    dtr = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
    dval = xgb.DMatrix(X_val, label=y_val, weight=w_val)
    dtest_fold = xgb.DMatrix(X_test_proc)
    
    model_xgb = xgb.train({
        'objective': 'reg:absoluteerror', 'tree_method': 'hist', 'device': 'cpu',
        'learning_rate': 0.017, 'max_depth': 10, 'min_child_weight': 25,
        'subsample': 0.9, 'colsample_bytree': 0.76, 'reg_alpha': 4.0, 'reg_lambda': 3.0
    }, dtr, num_boost_round=3000, evals=[(dval, 'val')], early_stopping_rounds=100, verbose_eval=False)
    
    oof_xgb[val_idx] = model_xgb.predict(dval)
    test_xgb[:, fold] = model_xgb.predict(dtest_fold)
    
    # LightGBM
    train_set = lgb.Dataset(X_tr, label=y_tr, weight=w_tr)
    val_set = lgb.Dataset(X_val, label=y_val, weight=w_val, reference=train_set)
    model_lgb = lgb.train({
        'objective': 'mae', 'metric': 'mae', 'learning_rate': 0.018, 'num_leaves': 128,
        'min_data_in_leaf': 30, 'feature_fraction': 0.78, 'bagging_fraction': 0.85,
        'lambda_l1': 1.5, 'lambda_l2': 4.0, 'verbosity': -1, 'seed': 42
    }, train_set, num_boost_round=3500, valid_sets=[val_set], callbacks=[lgb.early_stopping(100)])
    
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    test_lgb[:, fold] = model_lgb.predict(X_test_proc)
    
    # HistGB 
    model_hist = HistGradientBoostingRegressor(
        loss='absolute_error', learning_rate=0.025, max_iter=1200, max_depth=10,
        min_samples_leaf=25, l2_regularization=5.0, random_state=42+fold
    )
    model_hist.fit(X_tr, y_tr, sample_weight=w_tr)
    oof_hist[val_idx] = model_hist.predict(X_val)
    test_hist[:, fold] = model_hist.predict(X_test_proc)


pred_xgb = test_xgb.mean(axis=1)
pred_lgb = test_lgb.mean(axis=1)
pred_hist = test_hist.mean(axis=1)


stack_oof = np.column_stack([oof_xgb, oof_lgb, oof_hist])
stack_test = np.column_stack([pred_xgb, pred_lgb, pred_hist])

meta = Ridge(alpha=0.5, random_state=42)
meta.fit(stack_oof, y, sample_weight=w)

final_pred = meta.predict(stack_test)


os.makedirs('v5_pro', exist_ok=True)
pd.DataFrame({'id': test_id, 'predict': final_pred}).to_csv(
    'v5_pro/submission_v5_pro_stacking.csv', sep=';', decimal='.', index=False
)

print("завершено!")

In [ ]:
def advanced_feature_engineering(df):
    df = df.copy()
    

    if 'avg_credit_turn_rur' in df.columns and 'avg_debet_turn_rur' in df.columns:
        df['credit_debit_ratio'] = df['avg_credit_turn_rur'] / (df['avg_debet_turn_rur'] + 1e-8)
        df['log_credit_ratio'] = np.log1p(df['credit_debit_ratio'])
        df['credit_to_total_turn'] = df['avg_credit_turn_rur'] / (df['avg_credit_turn_rur'] + df['avg_debet_turn_rur'] + 1e-8)
    

    cat_cols = [c for c in df.columns if 'by_category__amount__sum__' in c]
    if cat_cols:
        numeric_cat = df[cat_cols].select_dtypes(include=[np.number]).columns
        if len(numeric_cat) > 0:
            df['cat_spend_mean'] = df[numeric_cat].mean(axis=1)
            df['cat_spend_sum'] = df[numeric_cat].sum(axis=1)
            df['cat_spend_std'] = df[numeric_cat].std(axis=1)
            df['cat_spend_max'] = df[numeric_cat].max(axis=1)
            df['cat_spend_skew'] = df[numeric_cat].skew(axis=1)
    

    turn_cr = [c for c in df.columns if 'cr_' in c and 'turn' in c]
    turn_db = [c for c in df.columns if 'db_' in c and 'turn' in c]
    if turn_cr and turn_db:
        cr_num = df[turn_cr].select_dtypes(include=[np.number]).columns
        db_num = df[turn_db].select_dtypes(include=[np.number]).columns
        if len(cr_num) > 0:
            df['total_cr'] = df[cr_num].sum(axis=1, skipna=True)
        if len(db_num) > 0:
            df['total_db'] = df[db_num].sum(axis=1, skipna=True)
            if 'total_cr' in df.columns:
                df['net_money_flow'] = df['total_cr'] - df['total_db']
    

    limit_cols = [c for c in df.columns if 'max_limit' in c or 'limit' in c]
    if limit_cols:
        num_limits = df[limit_cols].select_dtypes(include=[np.number]).columns
        if len(num_limits) > 0:
            df['total_limit'] = df[num_limits].sum(axis=1, skipna=True)
            df['mean_limit'] = df[num_limits].mean(axis=1)
    

    if 'days_from_start' in df.columns:
        df['days_squared'] = df['days_from_start'] ** 2
        df['days_log'] = np.log1p(df['days_from_start'])
        if 'age' in df.columns:
            df['age_days'] = df['age'] * df['days_from_start']
            df['age_days_log'] = df['age'] * np.log1p(df['days_from_start'])
    

    salary_cols = [c for c in df.columns if 'salary' in c or 'income' in c]
    if salary_cols:
        num_sal = df[salary_cols].select_dtypes(include=[np.number]).columns
        if len(num_sal) > 0:
            df['salary_mean'] = df[num_sal].mean(axis=1)
    
    return df

In [ ]:



tscv = TimeSeriesSplit(n_splits=5)

oof_preds = np.zeros((len(X_proc), 3))
test_preds = np.zeros((len(X_test_proc), 3))

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_proc)):
    print(f"Fold {fold+1}")
    X_tr = X_proc.iloc[tr_idx]
    X_val = X_proc.iloc[val_idx]
    y_tr = y[tr_idx]
    y_val = y[val_idx]
    w_tr = w[tr_idx]
    w_val = w[val_idx]
    
    # XGBoost
    dtr = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
    dval = xgb.DMatrix(X_val, label=y_val, weight=w_val)
    dtst = xgb.DMatrix(X_test_proc)
    
    xgb_model = xgb.train({
        'objective': 'reg:absoluteerror',
        'tree_method': 'hist',
        'learning_rate': 0.012,
        'max_depth': 11,
        'min_child_weight': 20,
        'subsample': 0.88,
        'colsample_bytree': 0.78,
        'reg_alpha': 3.5,
        'reg_lambda': 4.0,
    }, dtr, 4000, evals=[(dval,'val')], early_stopping_rounds=120, verbose_eval=False)
    
    oof_preds[val_idx, 0] = xgb_model.predict(dval)
    test_preds[:, 0] += xgb_model.predict(dtst) / 5
    
    # LightGBM
    lgb_train = lgb.Dataset(X_tr, y_tr, weight=w_tr)
    lgb_val = lgb.Dataset(X_val, y_val, weight=w_val, reference=lgb_train)
    lgb_model = lgb.train({
        'objective': 'mae',
        'learning_rate': 0.015,
        'num_leaves': 150,
        'min_data_in_leaf': 25,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.85,
        'lambda_l1': 2.0,
        'lambda_l2': 5.0,
    }, lgb_train, 4500, valid_sets=[lgb_val], callbacks=[lgb.early_stopping(100)])
    
    oof_preds[val_idx, 1] = lgb_model.predict(X_val)
    test_preds[:, 1] += lgb_model.predict(X_test_proc) / 5
    
    # HistGB
    hist_model = HistGradientBoostingRegressor(
        loss='absolute_error', learning_rate=0.02, max_iter=1500, max_depth=12,
        min_samples_leaf=20, l2_regularization=6.0, random_state=42+fold
    ).fit(X_tr, y_tr, sample_weight=w_tr)
    
    oof_preds[val_idx, 2] = hist_model.predict(X_val)
    test_preds[:, 2] += hist_model.predict(X_test_proc) / 5

# Meta
meta = Ridge(alpha=0.3)
meta.fit(oof_preds, y, sample_weight=w)

final_pred = meta.predict(test_preds)


pd.DataFrame({'id': test_id, 'predict': final_pred}).to_csv('submission_v6_ultra.csv', sep=';', decimal='.', index=False)


Запускаем V6 Ultra Stacking...
Fold 1
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[504]	valid_0's l1: 68829.5
Fold 2
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1188]	valid_0's l1: 65386.1
Fold 3
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[741]	valid_0's l1: 64218.4
Fold 4
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1728]	valid_0's l1: 59913.3
Fold 5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2201]	valid_0's l1: 61110.5
V6 Ultra готов!


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
from scipy.optimize import minimize
import os

print(" V7 ")


tscv = TimeSeriesSplit(n_splits=5)

oof_cb = np.zeros(len(X_proc))
oof_xgb = np.zeros(len(X_proc))
oof_lgb = np.zeros(len(X_proc))

test_cb = np.zeros((len(X_test_proc), 5))
test_xgb = np.zeros((len(X_test_proc), 5))
test_lgb = np.zeros((len(X_test_proc), 5))

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_proc)):
    print(f"\n=== Fold {fold+1}/5 ===")
    X_tr = X_proc.iloc[tr_idx]
    X_val = X_proc.iloc[val_idx]
    y_tr = y[tr_idx]
    y_val = y[val_idx]
    w_tr = w[tr_idx]
    w_val = w[val_idx]
    
    # CatBoost
    train_pool = Pool(X_tr, y_tr, weight=w_tr, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, weight=w_val, cat_features=cat_features)
    
    cb = CatBoostRegressor(
        iterations=6000,
        learning_rate=0.017,
        depth=10,
        l2_leaf_reg=5,
        loss_function='MAE',
        random_seed=42,
        verbose=300,
        early_stopping_rounds=200,
        task_type='CPU',
        border_count=254
    )
    cb.fit(train_pool, eval_set=val_pool)
    
    oof_cb[val_idx] = cb.predict(X_val)
    test_cb[:, fold] = cb.predict(X_test_proc)
    
    # XGBoost 
    dtr = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
    dval = xgb.DMatrix(X_val, label=y_val, weight=w_val)
    dtst = xgb.DMatrix(X_test_proc)
    
    xgb_params = {
        'objective': 'reg:absoluteerror',
        'tree_method': 'hist',
        'learning_rate': 0.014,
        'max_depth': 11,
        'min_child_weight': 20,
        'subsample': 0.87,
        'colsample_bytree': 0.78,
        'reg_alpha': 4.0,
        'reg_lambda': 5.0,
    }
    xgb_model = xgb.train(xgb_params, dtr, 4500, evals=[(dval,'val')], 
                         early_stopping_rounds=150, verbose_eval=False)
    
    oof_xgb[val_idx] = xgb_model.predict(dval)
    test_xgb[:, fold] = xgb_model.predict(dtst)
    
    # LightGBM 
    lgb_train = lgb.Dataset(X_tr, y_tr, weight=w_tr)
    lgb_val = lgb.Dataset(X_val, y_val, weight=w_val, reference=lgb_train)
    
    lgb_model = lgb.train({
        'objective': 'mae',
        'learning_rate': 0.015,
        'num_leaves': 200,
        'min_data_in_leaf': 20,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.85,
        'lambda_l1': 2.5,
        'lambda_l2': 6.0,
        'verbosity': -1
    }, lgb_train, 5000, valid_sets=[lgb_val], callbacks=[lgb.early_stopping(120)])
    
    oof_lgb[val_idx] = lgb_model.predict(X_val)
    test_lgb[:, fold] = lgb_model.predict(X_test_proc)


pred_cb = test_cb.mean(axis=1)
pred_xgb = test_xgb.mean(axis=1)
pred_lgb = test_lgb.mean(axis=1)


def blend_loss(weights):
    blended = weights[0]*pred_cb + weights[1]*pred_xgb + weights[2]*pred_lgb
    return np.sum(w * np.abs(y - (weights[0]*oof_cb + weights[1]*oof_xgb + weights[2]*oof_lgb))) / w.sum()

init_weights = [0.4, 0.35, 0.25]
bounds = [(0,1), (0,1), (0,1)]

from scipy.optimize import differential_evolution
result = differential_evolution(blend_loss, bounds, workers=1, tol=1e-5)

best_weights = result.x
print("Лучшие веса:", best_weights.round(4))

final_pred = best_weights[0]*pred_cb + best_weights[1]*pred_xgb + best_weights[2]*pred_lgb


os.makedirs('v7_ultra', exist_ok=True)

pd.DataFrame({'id': test_id, 'predict': final_pred}).to_csv(
    'v7_ultra/submission_v7_ultra.csv', sep=';', decimal='.', index=False
)


pd.DataFrame({'id': test_id, 'predict': pred_cb}).to_csv('v7_ultra/catboost.csv', sep=';', decimal='.', index=False)
pd.DataFrame({'id': test_id, 'predict': pred_xgb}).to_csv('v7_ultra/xgboost.csv', sep=';', decimal='.', index=False)
pd.DataFrame({'id': test_id, 'predict': pred_lgb}).to_csv('v7_ultra/lightgbm.csv', sep=';', decimal='.', index=False)



=== V7 ULTRA — Максимально тяжёлая модель ===

=== Fold 1/5 ===
0:	learn: 130200.1892351	test: 130259.8474856	best: 130259.8474856 (0)	total: 337ms	remaining: 33m 41s
300:	learn: 71540.7250903	test: 90726.9474858	best: 90726.9474858 (300)	total: 2m	remaining: 37m 57s
600:	learn: 54464.4491730	test: 86513.0713254	best: 86513.0713254 (600)	total: 3m 47s	remaining: 34m 5s
900:	learn: 42674.7629330	test: 84096.5537408	best: 84096.5537408 (900)	total: 5m 30s	remaining: 31m 10s
1200:	learn: 36012.6049473	test: 83006.6628193	best: 83006.6628193 (1200)	total: 7m 13s	remaining: 28m 52s
1500:	learn: 31978.5863910	test: 82294.9282492	best: 82294.8668969 (1499)	total: 8m 56s	remaining: 26m 48s
1800:	learn: 29381.2112691	test: 82027.2752184	best: 82027.2752184 (1800)	total: 10m 39s	remaining: 24m 52s
2100:	learn: 27509.4130841	test: 81809.6510155	best: 81809.6510155 (2100)	total: 12m 22s	remaining: 22m 57s
2400:	learn: 26084.6502465	test: 81615.7341523	best: 81615.7341523 (2400)	total: 14m 5s	remai